# Expfit notes 3: General error, Jacobian, and Hessian

$E_m$ is the error for $m$ exponentials

As before, we'll use a shorthand 
\begin{align}
e_{ij} = e^{b_j x_i}
\end{align}
But for clarity, we'll omit the $i$ indices:
\begin{align}
e_j = e^{b_j x}
\end{align}

## Two exponentials

\begin{align}
E_2 &= \frac{1}{n}\sum_{i=1}^n \left( y - a - b_1 e_1 - b_2 e_2 \right)^2
\end{align}

### Jacobian $J_2$

The general pattern here is

\begin{align}
\frac{\partial}{\partial x_1} (x_1 - x_2)^2 
= \frac{\partial}{\partial x_1} (x_1^2 - 2 x_1 x_2)
= 2 (x_1 - x_2)
\end{align}
and
\begin{align}
\frac{\partial}{\partial x_2} (x_1 - f(x_2))^2 
= \frac{\partial}{\partial x_2} (f(x_2)^2 - 2 x_1 f(x_2))
= 2 (f(x_2) - x_1) \frac{\partial}{\partial x_2} f(x_2)
\end{align}
which holds for each term in the sum.

So, without expanding the quadratic term, we can write



\begin{align}
\frac{\partial E_2}{\partial a} &= \frac{2}{n}\sum_{i=1}^n a - y + b_1e_1 + b_2e_2
\end{align}

\begin{align}
\frac{\partial E_2}{\partial b_1} &= \frac{2}{n}\sum_{i=1}^n\left( a - y + b_1 e_1 + b_2 e_2 \right) e_1 \\
\frac{\partial E_2}{\partial b_2} &= \frac{2}{n}\sum_{i=1}^n\left( a - y + b_1 e_1 + b_2 e_2 \right) e_2
\end{align}

\begin{align}
\frac{\partial E_2}{\partial c_1} &= \frac{2b_1}{n}\sum_{i=1}^n \left( a - y + b_1 e_1 + b_2 e_2 \right) xe_1 \\
\frac{\partial E_2}{\partial c_1} &= \frac{2b_2}{n}\sum_{i=1}^n \left( a - y + b_1 e_1 + b_2 e_2 \right) xe_2
\end{align}

## General case

\begin{align}
E_m &= \frac{1}{n}\sum_{i=1}^n \left( y - a - \sum_{j=1}^m b_j e_j \right)^2 &&  \\
\end{align}

### Jacobian

\begin{align}
\frac{\partial E_m}{\partial a} &= \frac{2}{n}\sum_{i=1}^n \left( a - y + \sum_{j=1}^m b_je_j \right)
\end{align}

\begin{align}
\frac{\partial E_m}{\partial b_s} &= \frac{2}{n}\sum_{i=1}^n\left( a - y + \sum_{j=1}^m b_j e_j \right) e_s
\end{align}

\begin{align}
\frac{\partial E_m}{\partial c_s} &= \frac{2b_s}{n}\sum_{i=1}^n \left( a - y + \sum_{j=1}^m b_j e_j \right) xe_s
\end{align}

### Hessian

\begin{align}
\frac{\partial^2 E_2}{\partial a^2} &= 2 \\
\frac{\partial^2 E_2}{\partial b_r\,\partial a} &= \frac{2}{n}\sum^n e_r \\
\frac{\partial^2 E_2}{\partial c_r\,\partial a} &= \frac{2b_r}{n}\sum^n xe_r
\end{align}

\begin{align}
\frac{\partial^2 E_2}{\partial b_r\,\partial b_s} &= \frac{2}{n}\sum_{i=1}^n e_r e_s \\
\end{align}

\begin{align}
\frac{\partial^2 E_2}{\partial c_s\,\partial b_s} &= \frac{2}{n}\sum_{i=1}^n \left( a - y + b_s e_s + \sum_{j=1}^m b_j e_j \right) xe_s \\
r \neq s, \frac{\partial^2 E_2}{\partial c_r\,\partial b_s} &= \frac{2b_r}{n}\sum_{i=1}^n x e_r e_s
\end{align}

\begin{align}
r \neq s, \frac{\partial^2 E_2}{\partial c_r c_s} &= \frac{2b_r b_s}{n}\sum_{i=1}^n x^2 e_r e_s \\
\frac{\partial^2 E_2}{\partial c_s^2} &= \frac{2b_s}{n}\sum_{i=1}^n \left( a - y + b_se_s + \sum_{j=1}^m b_j e_j \right) x^2e_s
\end{align}

## Checking our workings

First, we compare against the single-exponential case.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

In [2]:
def f1(x, a, b, c):
    return a + b * np.exp(c * x)

a, b, c = 5, 5, -3
n = 100
x = np.linspace(0, 1, n)
y = f1(x, a, b, c)

a0, b0, c0 = a + 1, b - 1, c + 1
yo = f1(x, a0, b0, c0)

In [3]:
def mse_jac_hes_1(x, y, p):
    a, b, c = p
    m = 1 / n
    e = np.exp(c * x)
    f = a - y + b * e
    ef = e * f
    mse = m * np.sum(f * f)

    # Jacobian
    jac = np.array([
        2 * m * np.sum(f),
        2 * m * np.sum(ef),
        2 * m * np.sum(ef * x) * b
    ])

    # Hessian
    ex = e * x
    aex = (a - y + 2 * b * e) * ex
    hes = np.array([
        [2, 2 * m * np.sum(e), 2 * b * m * np.sum(ex)],
        [0, 2 * m * np.sum(e * e), 2 * m * np.sum(aex)],
        [0, 0, 2 * m * b * np.sum(x * aex)],
    ])
    hes[1, 0] = hes[0, 1]
    hes[2, 0] = hes[0, 2]
    hes[2, 1] = hes[1, 2]
    
    return mse, jac, hes


In [4]:
def mse_jac_hes(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2

    # Unpack
    p = np.asarray(p)
    a = p[0]
    bs = p[1::2].reshape((m, 1))        # (m, 1)
    cs = p[2::2].reshape((m, 1))        # (m, 1)

    # MSE
    n = len(x)
    ninv2 = 2 / n
    es = np.exp(cs * x)                 # (m, n)  e^(cx)
    bes = bs * es                       # (m, n) be^(cx)
    fs = a - y + np.sum(bes, axis=0)    # (n, ) a - y + sum_j(be^(cx))
    mse = np.sum(fs**2) / n

    # Jacobian
    jac = np.zeros(d)
    xes = es * x
    jac[0] = ninv2 * np.sum(fs)
    jac[1::2] = ninv2 * np.sum(fs * es, axis=1)
    jac[2::2] = ninv2 * np.sum(fs * xes, axis=1) * bs.T

    # Hessian
    hes = np.zeros((d, d))
    # aa, ab, ac
    hes[0, 0] = 2
    hes[0, 1::2] = hes[1::2, 0] = ninv2 * np.sum(es, axis=1)
    hes[0, 2::2] = hes[2::2, 0] = ninv2 * np.sum(xes, axis=1) * bs.T
    for i in range(m):
        # bi^2, ci^2, and bi*ci
        hes[1 + 2 * i, 1 + 2 * i] = ninv2 * np.sum(es[i]**2)
        hes[2 + 2 * i, 2 + 2 * i] = ninv2 * np.sum((fs + bes[i]) * xes[i] * x) * bs[i, 0]
        hes[1 + 2 * i, 2 + 2 * i] = hes[2 + 2 * i, 1 + 2 * i] = \
            ninv2 * np.sum((fs + bes[i]) * xes[i])
        for j in range(i + 1, m):
            # bi*bj, ci*cj, bi*cj, bj*ci
            hes[1 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(es[i] * es[j])
            hes[2 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(xes[i] * xes[j]) * bs[i, 0] * bs[j, 0]
            hes[1 + 2 * i, 2 + 2 * j] = hes[2 + 2 * j, 1 + 2 * i] = \
                ninv2 * np.sum(xes[i] * es[j]) * bs[j, 0]
            hes[2 + 2 * i, 1 + 2 * j] = hes[1 + 2 * j, 2 + 2 * i] = \
                ninv2 * np.sum(xes[i] * es[j]) * bs[i, 0]

    return mse, jac, hes


m1, j1, h1 = mse_jac_hes_1(x, y, (a0, b0, c0))
m2, j2, h2 = mse_jac_hes(x, y, (a0, b0, c0))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
print(h1)
print(h2)

1.4223235997383419
1.4223235997383419

[2.28117017 0.83471883 1.46180251]
[2.28117017 0.83471883 1.46180251]

[[2.         0.86740054 1.18144538]
 [0.86740054 0.49618302 0.81578101]
 [1.18144538 0.81578101 1.60402956]]
[[2.         0.86740054 1.18144538]
 [0.86740054 0.49618302 0.81578101]
 [1.18144538 0.81578101 1.60402956]]


In [5]:
print(m1 - m2)
print(j1 - j2)
print(h1 - h2)

0.0
[0. 0. 0.]
[[0. 0. 0.]
 [0. 0. 0.]
 [0. 0. 0.]]


Now let's build a finite-difference version

In [6]:
def mse(x, y, p):
    d = len(p)
    assert (d - 1) % 2 == 0 and d > 1
    m = (d - 1) // 2
    p = np.asarray(p)    
    bs = p[1::2].reshape((m, 1))        # (m, 1)
    cs = p[2::2].reshape((m, 1))        # (m, 1)
    return np.sum((p[0] - y + np.sum(bs * np.exp(cs * x), axis=0))**2) / len(x)


def mse_jac_fd(x, y, p, dp):
    e = mse(x, y, p)
    jac = np.zeros(len(p))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        jac[i] = (mse(x, y, q) - e) / dp
    return e, jac
    
    
def mse_jac_hes_fd(x, y, p, dp=1e-5):
    d = len(p)
    m = (d - 1) // 2
    mse, jac = mse_jac_fd(x, y, p, dp)
    
    hes = np.zeros((d, d))
    p = np.array(p, dtype=float)
    for i in range(len(p)):
        q = np.copy(p)
        q[i] += dp
        hes[i] = (mse_jac_fd(x, y, q, dp)[1] - jac) / dp   
    return mse, jac, hes


m1, j1, h1 = mse_jac_hes_1(x, y, (a0, b0, c0))
m3, j3, h3 = mse_jac_hes_fd(x, y, (a0, b0, c0))
print(m1)
print(m3)
print()
print(j1)
print(j3)
print()
print(h1)
print(h3)

1.4223235997383419
1.4223235997383419

[2.28117017 0.83471883 1.46180251]
[2.28118017 0.83472131 1.46181053]

[[2.         0.86740054 1.18144538]
 [0.86740054 0.49618302 0.81578101]
 [1.18144538 0.81578101 1.60402956]]
[[1.99999795 0.86739727 1.18144383]
 [0.86739727 0.49617421 0.815783  ]
 [1.18144383 0.815783   1.60404356]]


And now we can do a proper comparison:

In [7]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0, 3, -7))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, c0, 3, -7))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
with np.printoptions(linewidth=120):
    print(h1)
    print(h2)

2.622100136408709
2.622100136408709

[3.15934834 1.52509551 1.75458651 0.62742162 0.2072607 ]
[3.15935834 1.52509799 1.75459485 0.62742238 0.20726102]

[[2.         0.86740054 1.18144538 0.29272606 0.12031674]
 [0.86740054 0.49618302 0.88897701 0.23012556 0.073196  ]
 [1.18144538 0.88897701 1.66882312 0.09759467 0.06479355]
 [0.29272606 0.23012556 0.09759467 0.15166407 0.09934221]
 [0.12031674 0.073196   0.06479355 0.09934221 0.06516193]]
[[2.00000905 0.86739949 1.18145493 0.2927214  0.12030821]
 [0.86739949 0.49617199 0.88898222 0.23011371 0.07319034]
 [1.18145493 0.88898222 1.66884728 0.09759304 0.06479706]
 [0.2927214  0.23011371 0.09759304 0.15165647 0.0993472 ]
 [0.12030821 0.07319034 0.06479706 0.0993472  0.06516121]]


This looks OK

In [8]:
with np.printoptions(linewidth=120):
    print((h1 - h2))

[[-9.04726494e-06  1.04994898e-06 -9.55722341e-06  4.65452443e-06  8.53026562e-06]
 [ 1.04994898e-06  1.10276501e-05 -5.20897592e-06  1.18536630e-05  5.65838418e-06]
 [-9.55722341e-06 -5.20897592e-06 -2.41655976e-05  1.62332278e-06 -3.50208481e-06]
 [ 4.65452443e-06  1.18536630e-05  1.62332278e-06  7.60849990e-06 -4.98410716e-06]
 [ 8.53026562e-06  5.65838418e-06 -3.50208481e-06 -4.98410716e-06  7.19027500e-07]]


In [9]:
m1, j1, h1 = mse_jac_hes(x, y, (a0, b0, c0, 3, -7, 1.1, 2.2))
m2, j2, h2 = mse_jac_hes_fd(x, y, (a0, b0, c0, 3, -7, 1.1, 2.2))
print(m1)
print(m2)
print()
print(j1)
print(j2)
print()
with np.printoptions(linewidth=120):
    print(h1 - h2)


36.63251670085841
36.63251670085841

[11.21471379  3.9606078   6.79114614  1.08861606  0.47753209 51.28690781
 42.84067694]
[11.21472379  3.96061028  6.79117164  1.08861681  0.47753293 51.28709297
 42.84100249]

[[-1.06746891e-04 -1.01090569e-04 -1.16138634e-04 -8.86042096e-05 -1.91309879e-04 -7.81470769e-05 -1.31864921e-04]
 [-1.01090569e-04 -1.31080897e-04 -6.69321829e-05 -9.02868552e-05 -6.09549973e-05 -2.01400710e-05 -8.39276674e-05]
 [-1.16138634e-04 -6.69321829e-05 -1.18206303e-04 -1.04958088e-04 -7.89972505e-05  2.42419323e-05 -1.85455394e-05]
 [-8.86042096e-05 -9.02868552e-05 -1.04958088e-04 -3.68004211e-05 -6.90688422e-05 -2.36016483e-05 -6.35313773e-06]
 [-1.91309879e-04 -6.09549973e-05 -7.89972505e-05 -6.90688422e-05 -8.21574868e-05 -7.54620540e-05 -8.73174064e-05]
 [-7.81470769e-05 -2.01400710e-05  2.42419323e-05 -2.36016483e-05 -7.54620540e-05 -5.76202352e-05 -8.22614795e-04]
 [-1.31864921e-04 -8.39276674e-05 -1.85455394e-05 -6.35313773e-06 -8.73174064e-05 -8.22614795e-04 